In [ ]:
!pip install evaluate
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from evaluate import load
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq


##EXERCISE 1

In [ ]:
def create_transformers_train_data(sentences, translations, tokenizer):
    inputs_en = tokenizer(sentences, max_length=10, truncation=True)

    with tokenizer.as_target_tokenizer():
        outputs_es = tokenizer(translations, max_length=10, truncation=True)

    data = Dataset.from_dict({'input_ids': inputs_en['input_ids'],
                              'attention_mask': inputs_en['attention_mask'],
                              'labels': outputs_es['input_ids']})
    return data



def train_transformer(model, train_loader, optimizer, epochs=5, device='cpu'):
    model = model.to(device)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(train_loader))
        print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt', sep='\t', header=None)
data.head()

,0,1
0,Style 1,Style 2
1,ever since joes has changed hands it's just go...,Ever since joes has changed hands it's gotten ...
2,there is definitely not enough room in that pa...,There is so much room in that part of the venue
3,so basically tasted watered down.,It didn't taste watered down at all.
4,she said she'd be back and disappeared for a f...,"She said she'd be back, and didn't disappear a..."


In [ ]:
from transformers import T5Tokenizer
from torch.utils.data import DataLoader

model_name = 't5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
train_dataset = create_transformers_train_data(
    data[0].values.tolist(),
    data[1].values.tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Train 't5-small'

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=data_collator)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = AdamW(model.parameters(), lr=0.001)

train_transformer(model, train_loader, optimizer, 5)

Epoch 1/5, Loss: 2.9638
Epoch 2/5, Loss: 2.3682
Epoch 3/5, Loss: 2.1460
Epoch 4/5, Loss: 2.0080
Epoch 5/5, Loss: 1.8848


In [ ]:
def decode_with_transformer(sentence, tokenizer, model, device='cpu'):
    model = model.to(device)
    model.eval()
    tokens = tokenizer([sentence], return_tensors='pt').to(device)
    out = model.generate(**tokens, max_length=10)

    with tokenizer.as_target_tokenizer():
        pred_sentence = tokenizer.decode(out[0], skip_special_tokens=True)

    return pred_sentence

Evaluation with T5

In [ ]:
predictions = []
references = []

for i in range(len(data)):
    sentence = data[0].iloc[i]
    pred = decode_with_transformer(sentence, tokenizer, model)
    ref = data[1].iloc[i]

    predictions.append(pred)
    references.append([ref])

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
bleu = load('bleu')

In [ ]:
bleu_results = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU score:", bleu_results["bleu"])

BLEU score: 0.26030830905522934


In [ ]:
bertscore = load('bertscore')

In [ ]:
from bert_score import score

P, R, F1 = score(
    predictions,
    [ref[0] for ref in references],
    lang="en",
    model_type="bert-base-uncased"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
print("\nPrecision:", P.mean().item())
print("Recall:", R.mean().item())
print("F1:", F1.mean().item())


Precision: 0.7280729413032532
Recall: 0.6566593050956726
F1: 0.6879682540893555


Train 'FLAN-T5'

In [ ]:
model_name = 'google/flan-t5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
train_dataset = create_transformers_train_data(
    data[0].values.tolist(),
    data[1].values.tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=data_collator)

In [ ]:
optimizer = AdamW(model.parameters(), lr=0.001)

train_transformer(model, train_loader, optimizer, 5)

Epoch 1/5, Loss: 2.6244
Epoch 2/5, Loss: 2.1051
Epoch 3/5, Loss: 1.8904
Epoch 4/5, Loss: 1.7187
Epoch 5/5, Loss: 1.5569


Evaluation with FLAN-T5

In [ ]:
predictions = []
references = []

for i in range(len(data)):
    sentence = data[0].iloc[i]
    pred = decode_with_transformer(sentence, tokenizer, model)
    ref = data[1].iloc[i]

    predictions.append(pred)
    references.append([ref])

In [ ]:
bleu_results = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU score:", bleu_results["bleu"])

P, R, F1 = score(
    predictions,
    [ref[0] for ref in references],
    lang="en",
    model_type="bert-base-uncased"
)


print("\nPrecision:", P.mean().item())
print("Recall:", R.mean().item())
print("F1:", F1.mean().item())


BLEU score: 0.2856855534564319

Precision: 0.7390393018722534
Recall: 0.6713624596595764
F1: 0.7008937001228333


##EXERCISE 2

In [ ]:
model_name = 't5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
instruction = 'Transform from Negative to Positive: '

In [ ]:
negative_sentences_with_ins = [f'{instruction}{s}' for s in data[0].values.tolist()]
negative_sentences_with_ins

['Transform from Negative to Positive: Style 1',
 "Transform from Negative to Positive: ever since joes has changed hands it's just gotten worse and worse.",
 'Transform from Negative to Positive: there is definitely not enough room in that part of the venue.',
 'Transform from Negative to Positive: so basically tasted watered down.',
 "Transform from Negative to Positive: she said she'd be back and disappeared for a few minutes.",
 "Transform from Negative to Positive: i can't believe how inconsiderate this pharmacy is.",
 'Transform from Negative to Positive: just left and took it off the bill.',
 "Transform from Negative to Positive: it isn't terrible, but it isn't very good either.",
 'Transform from Negative to Positive: definitely disappointed that i could not use my birthday gift!',
 "Transform from Negative to Positive: new owner, i heard - but i don't know the details.",
 'Transform from Negative to Positive: but it probably sucks too!',
 'Transform from Negative to Positive: 

In [ ]:
train_dataset = create_transformers_train_data(
    negative_sentences_with_ins,
    data[1].values.tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=data_collator)

In [ ]:
optimizer = AdamW(model.parameters(), lr=0.001)

train_transformer(model, train_loader, optimizer, 5)

Epoch 1/5, Loss: 5.2957
Epoch 2/5, Loss: 3.9484
Epoch 3/5, Loss: 3.6953
Epoch 4/5, Loss: 3.5291
Epoch 5/5, Loss: 3.4329


Evaluation with instructions

In [ ]:
predictions = []
references = []

for i in range(len(data)):
    sentence = data[0].iloc[i]
    pred = decode_with_transformer(sentence, tokenizer, model)
    ref = data[1].iloc[i]

    predictions.append(pred)
    references.append([ref])

In [ ]:
bleu_results = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU score:", bleu_results["bleu"])

P, R, F1 = score(
    predictions,
    [ref[0] for ref in references],
    lang="en",
    model_type="bert-base-uncased"
)


print("\nPrecision:", P.mean().item())
print("Recall:", R.mean().item())
print("F1:", F1.mean().item())


BLEU score: 0.10527758094456023

Precision: 0.5943880081176758
Recall: 0.5339434742927551
F1: 0.5605241656303406


Conclusion:

When the T5 model was trained without instructions, both BLEU and BERTScore were higher (BLEU ≈ 0.29, F1 ≈ 0.70), indicating that the model more closely reproduced the reference sentences in terms of words and structure.

With instructional fine-tuning, BLEU and BERTScore dropped significantly (BLEU ≈ 0.11, F1 ≈ 0.56). This is because the model now attempts to follow an abstract instruction (“transform a negative sentence into a positive one”), which allows for more diverse expressions and less direct overlap with the reference.

##EXERCISE 3

Tрансформација на реченици кои содржат негативен сентимент во реченици
кои содржат позитивен сентимент. (the same as the exercise 2)